# 03 Predict churn

Train on December 2010 to April 2011 and test on July to September 2011. Compare a simple recency rule and models on recency, orders and spend with gradient boosting on all 21 features. PR-AUC measures how well each method ranks churners first (a random ranking scores 0.56). Intervals come from resampling customers 1,000 times.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

from src.data import PROCESSED_DIR
from src.evaluation import (RAW_MEASURES, bootstrap_ci, fit_predict, make_boosting, make_logistic,
                            measure_columns, precision_at)
from src.labels import split_cutoffs, valid_cutoffs

features = pd.read_parquet(PROCESSED_DIR / "features.parquet")
train_cuts, test_cuts = split_cutoffs(valid_cutoffs(90), horizon_days=90, n_test=3)
train = features[features["cutoff"].isin(train_cuts)]
test = features[features["cutoff"].isin(test_cuts)].reset_index(drop=True)
y, customers = test["churn"].to_numpy(), test["customer_id"].to_numpy()

In [2]:
ALL = measure_columns(features)
scores = {
    "Recency rule": test["recency_days"].to_numpy(float),
    "Logistic, recency/orders/spend": fit_predict(make_logistic, RAW_MEASURES, train, test),
    "Boosting, recency/orders/spend": fit_predict(make_boosting, RAW_MEASURES, train, test),
    "Boosting, all features": fit_predict(make_boosting, ALL, train, test),
}

rows = []
for name, s in scores.items():
    lo, hi = bootstrap_ci(lambda ix: average_precision_score(y[ix], s[ix]), customers)
    rows.append({"method": name, "PR-AUC": round(average_precision_score(y, s), 3),
                 "95% CI": f"{lo:.3f} to {hi:.3f}", "churn rate in top 10%": f"{precision_at(y, s, 0.10):.0%}"})
pd.DataFrame(rows)

,method,PR-AUC,95% CI,churn rate in top 10%
0,Recency rule,0.738,0.721 to 0.753,78%
1,"Logistic, recency/orders/spend",0.788,0.773 to 0.802,84%
2,"Boosting, recency/orders/spend",0.787,0.771 to 0.802,84%
3,"Boosting, all features",0.794,0.779 to 0.808,86%


## Do the extra features help?

In [3]:
full, simple = scores["Boosting, all features"], scores["Boosting, recency/orders/spend"]
gain = lambda ix: average_precision_score(y[ix], full[ix]) - average_precision_score(y[ix], simple[ix])
lo, hi = bootstrap_ci(gain, customers)
print(f"PR-AUC gain from the extra features: {gain(np.arange(len(y))):+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")

test.assign(p_churn=full)[["customer_id", "cutoff", "churn", "spend_365d", "p_churn"]].to_parquet(
    PROCESSED_DIR / "test_predictions.parquet", index=False)

PR-AUC gain from the extra features: +0.007 (95% CI -0.002 to +0.016)


## Findings

The models clearly beat the recency rule: 86% of the customers flagged by gradient boosting churn, against 78% for the rule. The extra features add only +0.007 PR-AUC, and the interval includes zero, so this data can't show that they help. Recency, orders and spend carry most of the signal.